# Preprocessing: BPE tokenization

Dataset: `cardiffnlp/tweet_eval` (emotion), about 3.3k training tweets.

Preprocessing is kept minimal: punctuation is kept, and there is no stopword removal, stemming or lemmatisation (tweets are short, noisy and full of misspellings). We compare BPE tokenizers over a grid of hyperparameters:
- **lowercase** or keep the original case (capitals can signal intensity);
- **`vocab_size`**: number of tokens in the vocabulary;
- **`min_frequency`**: a pair of tokens is only merged if it appears at least this many times;
- **pre-tokenizer**: how text is split before BPE. The tutorial fixes `Metaspace` (blanks are replaced by `▁`; `!!!` or `:)` stay attached to words). We also try `Whitespace`, which splits punctuation from words. These are the only hyperparameters `part1.experiment.train_tokenizer` exposes.

The tokenizer is trained with `part1.experiment.train_tokenizer` from the tutorial.

In [35]:
import sys
from pathlib import Path

# Assuming the working directory is root/part1
sys.path.insert(0, str(Path.cwd().parent))

import part1
from datasets import load_dataset

dataset = load_dataset("cardiffnlp/tweet_eval", "emotion")
emotion_outcome_space = part1.clslib.NamedClasses(dataset["train"].features["label"].names)
example_tweet = dataset["train"][0]["text"]

print(dataset)
print(emotion_outcome_space)
print("Original:", example_tweet)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 3257
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1421
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 374
    })
})
0: anger, 1: joy, 2: optimism, 3: sadness
Original: “Worry is a down payment on a problem you may never have'.  Joyce Meyer.  #motivation #leadership #worry


## BPE tokenizer

The tokenizer is trained only on the training split.

In [36]:
from tokenizers.pre_tokenizers import Metaspace, Whitespace

PRETOKENIZERS = {"metaspace": Metaspace, "whitespace": Whitespace}


class BPEPreprocessor:
    """Optional lowercasing + BPE tokenization.

    Call `fit(train_texts)` first, then `tokenize(text)` returns the BPE tokens.
    """

    def __init__(self, lowercase=True, vocab_size=5000, min_frequency=2, pretokenizer="metaspace"):
        self.lowercase = lowercase
        self.vocab_size = vocab_size
        self.min_frequency = min_frequency
        self.pretokenizer = pretokenizer
        self.tokenizer = None

    def fit(self, train_texts):
        corpus = [t.lower() for t in train_texts] if self.lowercase else list(train_texts)
        self.tokenizer = part1.experiment.train_tokenizer(
            corpus, self.vocab_size, self.min_frequency, pretokenizer=PRETOKENIZERS[self.pretokenizer]()
        )
        return self

    def tokenize(self, text):
        return self.tokenizer.encode(text.lower() if self.lowercase else text).tokens

    def __len__(self):
        """Number of tokens in the learned vocabulary."""
        return self.tokenizer.get_vocab_size()

## Hyperparameter grid

Every combination below is trained on the training split only.

In [37]:
import itertools
from tqdm.auto import tqdm

GRID = {
    "lowercase": [True, False],
    "vocab_size": [500, 1000, 2000],
    "min_frequency": [2, 5],
    "pretokenizer": ["metaspace", "whitespace"],
}
configs = [dict(zip(GRID, values)) for values in itertools.product(*GRID.values())]
print(len(configs), "configurations")

train_texts = dataset["train"]["text"]
preprocessors = [BPEPreprocessor(**cfg).fit(train_texts) for cfg in tqdm(configs)]

print("Original:", example_tweet)
for cfg, prep in zip(configs, preprocessors):
    print(cfg, "->", prep.tokenize(example_tweet))


def find(**wanted):
    """Return the fitted preprocessor matching the given hyperparameters."""
    return next(p for c, p in zip(configs, preprocessors) if all(c[k] == v for k, v in wanted.items()))

24 configurations


  0%|          | 0/24 [00:00<?, ?it/s]

Original: “Worry is a down payment on a problem you may never have'.  Joyce Meyer.  #motivation #leadership #worry
{'lowercase': True, 'vocab_size': 500, 'min_frequency': 2, 'pretokenizer': 'metaspace'} -> ['▁', '“', 'w', 'or', 'ry', '▁is', '▁a', '▁d', 'ow', 'n', '▁p', 'ay', 'm', 'ent', '▁on', '▁a', '▁p', 'ro', 'ble', 'm', '▁you', '▁m', 'ay', '▁ne', 'ver', '▁have', "'", '.', '▁', '\xa0', 'j', 'o', 'y', 'ce', '▁me', 'y', 'er', '.', '▁', '▁#m', 'ot', 'i', 'v', 'at', 'ion', '▁#', 'le', 'ad', 'ers', 'h', 'i', 'p', '▁#', 'w', 'or', 'ry']
{'lowercase': True, 'vocab_size': 500, 'min_frequency': 2, 'pretokenizer': 'whitespace'} -> ['“', 'wor', 'ry', 'is', 'a', 'do', 'wn', 'p', 'ay', 'm', 'ent', 'on', 'a', 'pro', 'ble', 'm', 'you', 'm', 'ay', 'ne', 'ver', 'have', "'", '.', 'jo', 'y', 'ce', 'me', 'y', 'er', '.', '#', 'mo', 'ti', 'v', 'ation', '#', 'le', 'ad', 'er', 'sh', 'i', 'p', '#', 'wor', 'ry']
{'lowercase': True, 'vocab_size': 500, 'min_frequency': 5, 'pretokenizer': 'metaspace'} -> ['▁', '

## Inspecting every vocabulary

- **longest tokens**: whole words, hashtags or emojis it managed to keep in one piece;
- **first learned tokens**: ids right after the initial alphabet are the earliest (most frequent) merges;
- **tokens with symbols**: how punctuation and hashtags are handled.

In [38]:
import statistics

clean = lambda t: t.lstrip("\u2581")  # drop the Metaspace marker when measuring length

for cfg, prep in zip(configs, preprocessors):
    vocab = prep.tokenizer.get_vocab()  # token -> id
    by_id = sorted(vocab, key=vocab.get)

    print("=" * 100)
    print(cfg)
    lengths = [len(clean(t)) for t in vocab]
    print(f"token length: mean={statistics.mean(lengths):.2f}  median={statistics.median(lengths)}  max={max(lengths)}")
    print("longest tokens:", sorted(vocab, key=lambda t: -len(clean(t)))[:25])
    print("tokens with symbols:", [t for t in by_id if len(clean(t)) > 1 and not clean(t).replace("'", "").isalnum()][:30])

{'lowercase': True, 'vocab_size': 500, 'min_frequency': 2, 'pretokenizer': 'metaspace'}
token length: mean=1.54  median=1.0  max=5
longest tokens: ['▁@user', "▁don't", '▁about', '[UNK]', 'ight', 'king', '▁like', "▁it's", '▁your', '▁just', '▁this', '▁what', 'ress', 'ould', '▁when', 'user', '▁that', '▁they', '▁have', '▁some', 'ally', '▁with', 'ver', 'ate', '▁but']
tokens with symbols: ['[UNK]', '▁@user', '..', '▁#s', '\\n', '!!', '...', '▁#b', '▁#h', '▁#f', '▁#m']
{'lowercase': True, 'vocab_size': 500, 'min_frequency': 2, 'pretokenizer': 'whitespace'}
token length: mean=1.68  median=1.0  max=6
longest tokens: ['people', 'thing', '[UNK]', 'about', 'ation', 'press', 'with', 'user', 'this', 'your', 'ther', 'king', 'just', 'will', 'like', 'they', 'when', 'from', 'tion', 'what', 'pres', 'peop', 'have', 'ever', 'ight']
tokens with symbols: ['[UNK]', '..', '!!', '...']
{'lowercase': True, 'vocab_size': 500, 'min_frequency': 5, 'pretokenizer': 'metaspace'}
token length: mean=1.54  median=1.0  ma

## Default tokenizer

Shared configuration to start from: lowercase, `Whitespace` pre-tokenizer, 2000 tokens, `min_frequency=2`. Use `default_prep` to tokenize new text, and `preprocessed` (train/validation/test) for the already tokenized dataset with its labels.

In [ ]:
default_prep = BPEPreprocessor(lowercase=True, vocab_size=2000, min_frequency=2, pretokenizer="whitespace").fit(train_texts)
print(len(default_prep), "tokens")
print(default_prep.tokenize(example_tweet))

# Whole dataset, preprocessed: tokens and labels per split (same fitted tokenizer for all splits)
preprocessed = {
    split: {
        "tokens": [default_prep.tokenize(t) for t in dataset[split]["text"]],
        "labels": dataset[split]["label"],
    }
    for split in dataset
}
print({split: len(d["tokens"]) for split, d in preprocessed.items()})
print(preprocessed["train"]["tokens"][0], preprocessed["train"]["labels"][0])

2000 tokens
['“', 'worry', 'is', 'a', 'down', 'pay', 'ment', 'on', 'a', 'problem', 'you', 'may', 'never', 'have', "'", '.', 'joy', 'ce', 'me', 'y', 'er', '.', '#', 'motiv', 'ation', '#', 'lead', 'er', 'ship', '#', 'worry']
{'train': 3257, 'test': 1421, 'validation': 374}
['#', 'de', 'pp', 'ress', 'ion', 'is', 'real', '.', 'part', 'n', 'ers', 'w', '/', '#', 'depress', 'ed', 'people', 'tru', 'ly', 'dont', 'understand', 'the', 'dep', 'th', 'in', 'which', 'they', 'af', 'fect', 'us', '.', 'add', 'in', '#', 'anxiety', '&', 'amp', ';', 'makes', 'it', 'worse'] 2
